# Actividad en clase: Construir y evaluar un modelo de regresión lineal (scikit-learn)

**Fundamentos para IA · NRC 94103 · Semana 7**

**Dataset de trabajo:** `StudentsPerformance.csv` (1000 registros de estudiantes).

En [`01_Regresion_lineal_conceptos_basicos.md`](01_Regresion_lineal_conceptos_basicos.md) y en
[`Taller_02_Regresion_lineal_conceptos_basicos.md`](Taller_02_Regresion_lineal_conceptos_basicos.md) ya
calculaste `b0` y `b1` a mano y con `scipy.stats.linregress`, y evaluaste el modelo **sobre los mismos 1000
estudiantes** que se usaron para ajustarlo.

En esta actividad das el paso que falta y que es central en cualquier proyecto real de IA: separar los datos
en un conjunto de **entrenamiento** (para ajustar el modelo) y uno de **prueba** (para evaluarlo con
estudiantes que el modelo *nunca vio*), usando **`scikit-learn`**, la librería de machine learning más usada
en la industria.

**Objetivos de la actividad:**

1. Entrenar un modelo de regresión lineal con `scikit-learn`.
2. Separar los datos en entrenamiento y prueba, y entender por qué importa.
3. Evaluar el modelo con **R², MAE y RMSE** sobre datos no vistos.
4. Visualizar predicciones y **residuos**.
5. Extender el modelo a **regresión múltiple**.
6. Predecir la nota de un estudiante nuevo hipotético.

Trabajen en parejas o grupos pequeños, en un *notebook* (Colab, Jupyter o Anaconda).

**Configuración inicial** (ejecuta esto una sola vez, al comienzo):

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv("StudentsPerformance.csv")
df.head()

---

## Ejercicio 1 — Preparar los datos (`X`, `y`)

A diferencia de `scipy.stats.linregress` (que recibía dos columnas sueltas), `scikit-learn` espera:

- `X`: una matriz **2D** con las variables predictoras (aunque sea una sola columna).
- `y`: un vector **1D** con la variable a predecir.

$$\hat{y} = b_0 + b_1 x \qquad \text{(la misma ecuación de siempre, ahora ajustada con scikit-learn)}$$

In [ ]:
X = df[["reading score"]]   # doble corchete -> DataFrame de 1 columna (2D), lo que pide sklearn
y = df["____"]               # writing score, como una Serie (1D)

print(f"Forma de X: {____}")   # (1000, 1)
print(f"Forma de y: {____}")   # (1000,)

**Preguntas:**

a) ¿Por qué `X` necesita ser una matriz 2D aunque solo tenga una columna, mientras que `y` puede ser un
vector 1D?
b) ¿Qué pasaría con la forma de `X` si en vez de `df[["reading score"]]` hubieras escrito
`df["reading score"]` (un solo corchete)?

---

## Ejercicio 2 — Dividir en entrenamiento y prueba

Vamos a reservar una parte de los estudiantes (por ejemplo, el 20%) como **conjunto de prueba**: el modelo
nunca los va a ver durante el entrenamiento, así que sirven para simular "estudiantes nuevos" y medir qué
tan bien generaliza el modelo.

`random_state=42` fija la "semilla" aleatoria: así, cada vez que ejecutes esta celda, obtienes exactamente
la misma división (útil para que los resultados sean reproducibles y comparables entre grupos).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=____, random_state=42   # 20% de los datos para prueba
)

print(f"Estudiantes de entrenamiento: {len(X_train)}")
print(f"Estudiantes de prueba: {len(____)}")

**Preguntas:**

a) ¿Qué problema habría si evaluáramos el modelo con los **mismos** datos que usamos para entrenarlo, en
vez de reservar un conjunto de prueba aparte?
b) ¿Por qué es importante fijar `random_state` en vez de dejar que la división sea distinta cada vez que se
ejecuta el código?

---

## Ejercicio 3 — Entrenar el modelo

`LinearRegression()` crea un modelo vacío; `.fit(X_train, y_train)` es el paso que realmente **calcula**
`b0` y `b1` a partir del conjunto de entrenamiento (por dentro, usa la misma idea de mínimos cuadrados que
ya conoces de `01_...md`).

In [ ]:
modelo = LinearRegression()
modelo.____(X_train, y_train)   # ajusta (entrena) el modelo con los datos de entrenamiento

print(f"b1 (pendiente) = {modelo.coef_[0]:.4f}")
print(f"b0 (intercepto) = {modelo.____:.4f}")

**Preguntas:**

a) Compara estos coeficientes con los del `01_...md` / `Taller_02_...` (`b1 ≈ 0.9935`, `b0 ≈ -0.6676`,
calculados con los 1000 estudiantes). ¿Son iguales, parecidos o muy distintos? ¿Por qué tendría sentido que
no sean exactamente iguales?
b) ¿Qué método de `scikit-learn` fue el que realmente "aprendió" los coeficientes a partir de los datos?

---

## Ejercicio 4 — Predecir y visualizar

Ahora usamos el modelo ya entrenado para predecir `writing score` sobre el conjunto de **prueba**
(estudiantes que el modelo nunca vio durante el entrenamiento).

In [ ]:
y_pred_test = modelo.predict(____)   # predicciones sobre X_test

plt.scatter(X_test, y_test, alpha=0.4, label="Real")
plt.plot(X_test, y_pred_test, color="red", linewidth=2, label="Predicción del modelo")
plt.xlabel("Reading score")
plt.ylabel("Writing score")
plt.title("Predicciones del modelo sobre datos de prueba (nunca vistos)")
plt.legend()
plt.show()

**Preguntas:**

a) Mirando el gráfico, ¿las predicciones (línea roja) siguen razonablemente bien a los puntos reales de
`X_test`, aunque el modelo nunca los vio durante el entrenamiento?
b) ¿Por qué la "predicción del modelo" se ve como una línea recta y no como una nube de puntos, aunque se
está graficando sobre `X_test`?

---

## Ejercicio 5 — Evaluar el modelo (R², MAE, RMSE)

Ya viste el **R²** en `01_...md`. Aquí lo calculamos con `scikit-learn` sobre el conjunto de **prueba**
(no sobre los datos de entrenamiento), y lo acompañamos de dos métricas en la misma escala de las notas:

$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i| \qquad\qquad RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

- **MAE** (error absoluto medio): en promedio, ¿cuántos puntos se equivoca el modelo, sin importar si
  predice de más o de menos?
- **RMSE** (raíz del error cuadrático medio): parecido al MAE, pero castiga más fuerte los errores grandes
  (por el cuadrado antes de la raíz).

In [ ]:
r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, ____))

print(f"R²   (prueba) = {____:.4f}")
print(f"MAE  (prueba) = {mae_test:.4f}")
print(f"RMSE (prueba) = {rmse_test:.4f}")

# Comparamos con el desempeño sobre los datos de ENTRENAMIENTO
y_pred_train = modelo.predict(X_train)
r2_train = r2_score(y_train, y_pred_train)
print(f"\nR² (entrenamiento) = {____:.4f}")

**Preguntas:**

a) ¿El R² de prueba es parecido al R² de entrenamiento, o muy distinto? ¿Qué te diría una diferencia muy
grande entre ambos (por ejemplo, R² de entrenamiento mucho más alto que el de prueba)?
b) En la escala original de las notas, ¿cómo interpretarías el valor de RMSE que obtuviste (por ejemplo,
"en promedio el modelo se equivoca por ± ___ puntos")?

---

## Ejercicio 6 — Analizar los residuos

El **residuo** de cada estudiante es la diferencia entre su nota real y la predicha por el modelo
($y_i - \hat{y}_i$). Si el modelo captura bien la relación, los residuos deberían verse como una nube
"aleatoria" alrededor de cero, sin ningún patrón claro.

In [ ]:
residuos_test = y_test - y_pred_test

plt.scatter(y_pred_test, residuos_test, alpha=0.4)
plt.axhline(0, color="____", linestyle="--")   # línea horizontal de referencia en y=0
plt.xlabel("Predicción del modelo")
plt.ylabel("Residuo (real - predicho)")
plt.title("Gráfico de residuos (conjunto de prueba)")
plt.show()

**Preguntas:**

a) ¿Los residuos se ven distribuidos de forma pareja alrededor de cero, o notas algún patrón (por ejemplo,
que crezcan o se abran como un abanico a medida que la predicción aumenta)?
b) ¿Qué problema indicaría un residuo muy grande para un estudiante en particular (por ejemplo, +25
puntos)?

---

## Ejercicio 7 — Regresión múltiple con scikit-learn

Igual que en el Ejercicio 4 del taller anterior, agregamos `math score` como segunda variable predictora:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2$$

y comparamos su R² de prueba contra el del modelo simple, para ver si agregar la variable realmente ayuda a
**generalizar** mejor (no solo a ajustar mejor los datos de entrenamiento).

In [ ]:
X_multi = df[["reading score", "math score"]]
y_multi = df["writing score"]

X_multi_train, X_multi_test, y_multi_train, y_multi_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

modelo_multi = LinearRegression()
modelo_multi.____(X_multi_train, y_multi_train)

y_multi_pred = modelo_multi.predict(____)
r2_multi = r2_score(y_multi_test, y_multi_pred)

print(f"Coeficientes -> reading: {modelo_multi.coef_[0]:.4f}, math: {modelo_multi.coef_[1]:.4f}")
print(f"Intercepto -> {modelo_multi.intercept_:.4f}")
print(f"\nR² (prueba, modelo múltiple) = {____:.4f}")
print(f"R² (prueba, modelo simple)   = {r2_test:.4f}")

**Preguntas:**

a) ¿El R² de prueba mejora al agregar `math score`, se mantiene casi igual, o empeora? ¿Cómo se compara
con lo que ya habías visto en el Ejercicio 4 del `Taller_02_...` (con `statsmodels`, ajustado sobre los
1000 estudiantes completos)?
b) Si la mejora en R² fuera muy pequeña, ¿agregarías de todas formas `math score` al modelo final? Justifica
pensando en la relación costo (una variable más que recolectar) / beneficio (una mejora mínima).

---

## Ejercicio 8 — Predecir para un estudiante nuevo

Por último, usa el modelo múltiple ya entrenado para predecir la nota de escritura de un estudiante
hipotético que **no está en el dataset**, dando solo sus notas de lectura y matemáticas.

In [ ]:
estudiante_nuevo = pd.DataFrame({"reading score": [78], "math score": [82]})
prediccion_nueva = modelo_multi.predict(____)

print(f"Predicción de writing score -> {prediccion_nueva[0]:.2f}")

**Preguntas:**

a) ¿Por qué `estudiante_nuevo` se construye como un `DataFrame` con las mismas columnas (`"reading score"`,
`"math score"`) y en el mismo orden que `X_multi`, y no como una simple lista de números?
b) Cambia los valores de `reading score` y `math score` por otros de tu elección y vuelve a predecir. ¿La
predicción cambia en la dirección que esperarías (sube si subes las notas de entrada, baja si las bajas)?

---

## Cierre de la actividad

Repasando lo que hiciste hoy, conectado con `01_Regresion_lineal_conceptos_basicos.md`:

1. **Misma ecuación, otra manera de ajustarla**: `scikit-learn` calcula los mismos `b0`/`b1` que ya
   conocías, pero pensando siempre en términos de `X` (matriz 2D) y `y` (vector), el estándar que usan casi
   todas las librerías de machine learning.
2. **Entrenamiento vs. prueba**: separar los datos y evaluar solo sobre el conjunto de prueba es la
   diferencia central entre "ajustar una recta a un montón de puntos" y "construir un modelo que se pueda
   confiar para predecir datos nuevos" — la idea más importante de esta actividad.
3. **R², MAE y RMSE** son tres formas distintas de resumir qué tan bien predice el modelo; verlas juntas (y
   comparando entrenamiento vs. prueba) da una imagen más completa que mirar solo una métrica.
4. **Los residuos** son la primera herramienta de diagnóstico: si tienen un patrón claro, el modelo lineal
   probablemente se le está escapando algo.
5. **Regresión múltiple**: agregar variables no cambia la naturaleza del modelo (sigue siendo una
   combinación lineal de las entradas), y su beneficio real se mide en el conjunto de **prueba**, no en qué
   tanto "se ve mejor" sobre los datos de entrenamiento.

## Material relacionado

Conceptos sin código: [`01_Regresion_lineal_conceptos_basicos.md`](01_Regresion_lineal_conceptos_basicos.md).
Taller con `scipy`/`statsmodels` (hipótesis, p-values, Levene, variable dummy):
[`Taller_02_Regresion_lineal_conceptos_basicos.md`](Taller_02_Regresion_lineal_conceptos_basicos.md) /
[`Taller_02_Regresion_lineal_conceptos_basicos.ipynb`](Taller_02_Regresion_lineal_conceptos_basicos.ipynb).
Taller manual sin código (fórmulas a mano):
[`Taller_01_Estadistica_regresion_lineal.md`](Taller_01_Estadistica_regresion_lineal.md).